<a href="https://colab.research.google.com/github/mmomayck/GeoAI/blob/main/Hackathon_burned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Download the dataset (image and building footprints) from Zenodo
!mkdir Dataset
!wget  https://zenodo.org/records/16948350/files/23728930_15.tiff?download=1  -O Dataset/Image.tif
!wget  https://zenodo.org/records/16948350/files/BuildingFootPrint.gpkg?download=1 -O Dataset/GT.gpkg

# Install necessary packages
!pip install rasterio
!pip install dbfread

--2026-06-10 14:43:17--  https://zenodo.org/records/16948350/files/23728930_15.tiff?download=1
Resolving zenodo.org (zenodo.org)... 188.185.43.153, 137.138.52.235, 137.138.153.219, ...
Connecting to zenodo.org (zenodo.org)|188.185.43.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6765878 (6.5M) [image/tiff]
Saving to: ‘Dataset/Image.tif’

Dataset/Image.tif   100%[===================>]   6.45M   650KB/s    in 13s     

2026-06-10 14:43:30 (528 KB/s) - ‘Dataset/Image.tif’ saved [6765878/6765878]

--2026-06-10 14:43:30--  https://zenodo.org/records/16948350/files/BuildingFootPrint.gpkg?download=1
Resolving zenodo.org (zenodo.org)... 188.185.43.153, 137.138.52.235, 137.138.153.219, ...
Connecting to zenodo.org (zenodo.org)|188.185.43.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 958464 (936K) [application/octet-stream]
Saving to: ‘Dataset/GT.gpkg’

Dataset/GT.gpkg     100%[===================>] 936.00K   506KB/s    in 1.9s    


In [4]:
# Import the libraries
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio import features
from rasterio.plot import show
from rasterio.warp import calculate_default_transform, reproject, Resampling
import geopandas as gpd
from tqdm.notebook import tqdm
import folium
from folium import plugins

In [5]:
import os
import glob

folder_path = '/content/drive/MyDrive/Burned_S2'

file_list = glob.glob(os.path.join(folder_path, '*'))

print(f"พบไฟล์ทั้งหมด: {len(file_list)} ไฟล์\n")


for file_path in file_list:

    file_name = os.path.basename(file_path)
    print(file_name)
import re

def extract_date_from_filename(filename):
    match = re.search(r'S\d{1}[A-Z]_(\d{8})', filename)
    if match:
        return match.group(1)
    return None

# Create a list of (date, filename) tuples
files_with_dates = []
for file_path in file_list:
    file_name = os.path.basename(file_path)
    date_str = extract_date_from_filename(file_name)
    if date_str:
        files_with_dates.append((date_str, file_name))

# Sort the list by date
sorted_files_with_dates = sorted(files_with_dates, key=lambda x: x[0])

print("ไฟล์ที่เรียงตามวันที่:")
for date_str, file_name in sorted_files_with_dates:
    print(f"{date_str}: {file_name}")

พบไฟล์ทั้งหมด: 32 ไฟล์

Burned_S2_20260101_20260131_thailand.sbx
Burned_S2_20260101_20260131_thailand.sbn
Burned_S2_20260101_20260131_thailand.cpg
Burned_S2_20260101_20260131_thailand.shx
Burned_S2_20260101_20260131_thailand.prj
Burned_S2_20260101_20260131_thailand.dbf
Burned_S2_20260101_20260131_thailand.shp.xml
Burned_S2_20260101_20260131_thailand.shp
Burned_S2_20260201_20260228_thailand.prj
Burned_S2_20260201_20260228_thailand.cpg
Burned_S2_20260201_20260228_thailand.shp
Burned_S2_20260201_20260228_thailand.shx
Burned_S2_20260201_20260228_thailand.dbf
Burned_S2_20260201_20260228_thailand.shp.xml
Burned_S2_20260201_20260228_thailand.sbx
Burned_S2_20260201_20260228_thailand.sbn
Burned_S2_20260301_20260331_thailand.prj
Burned_S2_20260301_20260331_thailand.shx
Burned_S2_20260301_20260331_thailand.shp
Burned_S2_20260301_20260331_thailand.sbx
Burned_S2_20260301_20260331_thailand.sbn
Burned_S2_20260301_20260331_thailand.shp.xml
Burned_S2_20260301_20260331_thailand.cpg
Burned_S2_20260301_20

In [6]:
import geopandas as gpd

shapefile_path = '/content/drive/MyDrive/Burned_S2/Burned_S2_20260101_20260131_thailand.shp'

gdf = gpd.read_file(shapefile_path)

gdf.head()

/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Polygon' is converted to 'Polygon Z'
  return ogr_read(


,BURN,TB_IDN,TB_TN,AP_TN,PV_TN,PV_EN,RE_NESDB,RE_ROYIN,Area_rai,RP_CODE,RP_NAME,ClusterFin,name,LU_CODE,LU_Name,Area_Map,Shape_Leng,Shape_Area,Source,geometry
0,1,710705,ต.ชะแล,อ.ทองผาภูมิ,จ.กาญจนบุรี,Kanchanaburi,West,West,4.234471,1,ป่าอนุรักษ์,12,กลุ่มป่ารอบเขื่อนศรีนครินทร์,O000,อื่น ๆ,4,0.005463,5.689347e-07,GISTDA,"MULTIPOLYGON Z (((98.8233 14.84959 0, 98.82324..."
1,1,710706,ต.ห้วยเขย่ง,อ.ทองผาภูมิ,จ.กาญจนบุรี,Kanchanaburi,West,West,2.752067,1,ป่าอนุรักษ์,None,None,O000,อื่น ๆ,3,0.003636,3.695032e-07,GISTDA,"POLYGON Z ((98.59019 14.69326 0, 98.59019 14.6..."
2,1,710611,ต.ท่าตะคร้อ,อ.ท่าม่วง,จ.กาญจนบุรี,Kanchanaburi,West,West,0.052637,5,พื้นที่เกษตร,None,None,O000,อื่น ๆ,0,0.000481,7.043484e-09,GISTDA,"POLYGON Z ((99.69227 13.91523 0, 99.69227 13.9..."
3,1,710611,ต.ท่าตะคร้อ,อ.ท่าม่วง,จ.กาญจนบุรี,Kanchanaburi,West,West,0.406916,5,พื้นที่เกษตร,None,None,O000,อื่น ๆ,0,0.002872,5.444515e-08,GISTDA,"MULTIPOLYGON Z (((99.68428 13.8941 0, 99.68427..."
4,1,710611,ต.ท่าตะคร้อ,อ.ท่าม่วง,จ.กาญจนบุรี,Kanchanaburi,West,West,0.026425,5,พื้นที่เกษตร,None,None,O000,อื่น ๆ,0,0.000362,3.535867e-09,GISTDA,"POLYGON Z ((99.68317 13.90802 0, 99.68301 13.9..."


In [7]:
gdf.geometry

,geometry
0,"MULTIPOLYGON Z (((98.8233 14.84959 0, 98.82324..."
1,"POLYGON Z ((98.59019 14.69326 0, 98.59019 14.6..."
2,"POLYGON Z ((99.69227 13.91523 0, 99.69227 13.9..."
3,"MULTIPOLYGON Z (((99.68428 13.8941 0, 99.68427..."
4,"POLYGON Z ((99.68317 13.90802 0, 99.68301 13.9..."
...,...
47563,"MULTIPOLYGON Z (((101.80446 15.87247 0, 101.80..."
47564,"POLYGON Z ((101.80442 15.87247 0, 101.80449 15..."
47565,"POLYGON Z ((101.80442 15.87247 0, 101.80449 15..."
47566,"POLYGON Z ((101.88476 15.89345 0, 101.88476 15..."


In [8]:
print(f"ระบบพิกัด burned: {gdf.crs}")

ระบบพิกัด burned: EPSG:4326


In [9]:
# พื้นที่เพาะปลูกข้าวสำหรับจังหวัดกาญจนบุรี (เนื่องจาก 'เชียงใหม่' อาจจะไม่มีอยู่ในข้อมูลตัวอย่าง)
burned_cm = gdf[gdf["PV_TN"] == "จ.เชียงใหม่"].copy()

# แปลงคอลัมน์บางคอลัมน์เป็น String เพื่อป้องกัน Folium แปลง JSON Error (ยกเว้น 'geometry')
date_cols = [
    "AP_TN",	"PV_TN",	"PV_EN", "Shape_Leng",	"Shape_Area"
]
burned_cm[date_cols] = burned_cm[date_cols].astype(str)

# สร้างแผนที่ฐานออนไลน์
map_chiangmai = folium.Map(location=[18.79, 98.99], zoom_start=9, tiles="OpenStreetMap")

# เลือกคอลัมน์ที่จะแสดงใน Tooltip
tooltip_fields = [c for c in burned_cm.columns if c != "geometry"]

# ซ้อนทับเลเยอร์ Polygon
folium.GeoJson(
    burned_cm,
    style_function=lambda f: {
        "fillColor": "#e0115f",
        "color": "#700934",
        "weight": 0.6,
        "fillOpacity": 0.5
    },
    tooltip=folium.GeoJsonTooltip(
        fields=tooltip_fields,
        sticky=True
    )
).add_to(map_chiangmai)

map_chiangmai

##รวมไฟล์ PM 2.5 จากวันเป็นเดือน

In [11]:
import geopandas as gpd
import pandas as pd
import glob
import os

# กำหนด Path หลักที่เก็บโฟลเดอร์ของแต่ละเดือน
base_path = '/content/drive/MyDrive/PM25_Daily_Shapefiles'

months_to_process = {
    '01_January': '01',
    '02_February': '02',
    '03_March': '03',
    '04_April': '04'
}

all_months_gdfs = []

for month_folder, month_num in months_to_process.items():
    folder_path = os.path.join(base_path, month_folder)

    # ปรับ Pattern ให้ค้นหาไฟล์ตามเดือนนั้นๆ (เช่น PM25_2026-02-*.shp)
    file_pattern = os.path.join(folder_path, f"PM25_2026-{month_num}-*.shp")
    file_list = glob.glob(file_pattern)

    # เช็คว่ามีไฟล์ในเดือนนั้นไหม ป้องกัน Error
    if not file_list:
        print(f"ไม่พบไฟล์ในเดือน {month_folder}")
        continue

    print(f"กำลังประมวลผลเดือน {month_folder} (พบ {len(file_list)} ไฟล์)...")

    gdfs = []
    for file in file_list:
        gdf = gpd.read_file(file)
        gdfs.append(gdf)

    # รวมข้อมูลในเดือนนั้นเข้าด้วยกัน
    merged_monthly_gdf = pd.concat(gdfs, ignore_index=True)

    # ทำให้แน่ใจว่าผลลัพธ์ยังคงเป็น GeoDataFrame และมี CRS (ระบบพิกัด) เดิม
    merged_monthly_gdf = gpd.GeoDataFrame(merged_monthly_gdf, geometry='geometry', crs=gdfs[0].crs)

    # ตั้งชื่อและบันทึกไฟล์รายเดือน
    output_path = os.path.join(folder_path, f'merged_monthly_2026_{month_num}.shp')
    merged_monthly_gdf.to_file(output_path)

    print(f"บันทึกไฟล์ {output_path} สำเร็จ!\n")

    # เก็บข้อมูลของแต่ละเดือนไว้ใน List ใหญ่ (เพื่อนำไปทำ DBSCAN ต่อ)
    all_months_gdfs.append(merged_monthly_gdf)

print("ประมวลผลครบแล้ว")

กำลังประมวลผลเดือน 01_January (พบ 31 ไฟล์)...
บันทึกไฟล์ /content/drive/MyDrive/PM25_Daily_Shapefiles/01_January/merged_monthly_2026_01.shp สำเร็จ!

กำลังประมวลผลเดือน 02_February (พบ 28 ไฟล์)...
บันทึกไฟล์ /content/drive/MyDrive/PM25_Daily_Shapefiles/02_February/merged_monthly_2026_02.shp สำเร็จ!

กำลังประมวลผลเดือน 03_March (พบ 31 ไฟล์)...
บันทึกไฟล์ /content/drive/MyDrive/PM25_Daily_Shapefiles/03_March/merged_monthly_2026_03.shp สำเร็จ!

กำลังประมวลผลเดือน 04_April (พบ 4 ไฟล์)...
บันทึกไฟล์ /content/drive/MyDrive/PM25_Daily_Shapefiles/04_April/merged_monthly_2026_04.shp สำเร็จ!

ประมวลผลครบแล้ว


In [10]:
shapefile_path_jan = '/content/drive/MyDrive/PM25_Daily_Shapefiles/01_January/merged_monthly_2026_01.shp'

pm_jan = gpd.read_file(shapefile_path_jan)
pm_jan

,Date,PM25,Station,Lat,Long,geometry
0,2026/01/24 00:00:00,28.587500,Chiang Mai City Hall,18.836,98.970,POINT (98.97 18.836)
1,2026/01/24 00:00:00,28.587500,"Mae Rim, Chiang Mai",18.914,98.948,POINT (98.948 18.914)
2,2026/01/24 00:00:00,28.587500,"Doi Suthep, Chiang Mai",18.804,98.921,POINT (98.921 18.804)
3,2026/01/24 00:00:00,19.316667,"Chiang Dao, Chiang Mai",19.366,98.964,POINT (98.964 19.366)
4,2026/01/24 00:00:00,11.504167,Mae Hong Son Center,19.302,97.965,POINT (97.965 19.302)
...,...,...,...,...,...,...
243,2026/01/30 00:00:00,16.520833,"Chiang Dao, Chiang Mai",19.366,98.964,POINT (98.964 19.366)
244,2026/01/30 00:00:00,16.275000,Mae Hong Son Center,19.302,97.965,POINT (97.965 19.302)
245,2026/01/30 00:00:00,13.987500,"Pai, Mae Hong Son",19.358,98.440,POINT (98.44 19.358)
246,2026/01/30 00:00:00,26.945833,Chiang Rai Center,19.908,99.832,POINT (99.832 19.908)
